### Imports and loader


In [1]:
import json
from pathlib import Path

from typing import List, Dict, Any

In [2]:
DATA_PATH = Path(
    "../../legal-dataset/acts/consumer-protection-act-2019/final/v2-knowledge-cards.json"
)

print(DATA_PATH)
print(DATA_PATH.exists())

..\..\legal-dataset\acts\consumer-protection-act-2019\final\v2-knowledge-cards.json
True


In [5]:
with open(DATA_PATH, "r", encoding="utf-8") as file:
    knowledge_cards = json.load(file)

print(type(knowledge_cards))
print(len(knowledge_cards))

first_card = knowledge_cards[0]

print(first_card)

print("Concept ID:")
print(first_card.get("concept_id"))

print("\nConcept Type:")
print(first_card.get("concept_type"))

print("\nTitle:")
print(first_card.get("title"))

print("\nDescription:")
print(first_card.get("description"))

print("\nContent:")
print(first_card.get("content"))

print("\nRelated Concepts:")
print(first_card.get("related_concepts"))

print("\nSearch:")
print(first_card.get("search"))

print("\nMetadata:")
print(first_card.get("metadata"))

<class 'list'>
4147
{'concept_id': 'alias.accounts_and_audit_reports', 'concept_type': 'alias', 'title': 'Accounts and audit reports', 'description': 'Alternative names and search aliases for: Accounts and audit reports', 'content': {'aliases': ['accounting reports', 'audit records', 'financial statements']}, 'derived_from': [], 'related_concepts': ['evidence.accounts_audit_report'], 'search': {'keywords': ['accounts', 'audit', 'reports', 'alternative', 'names', 'search', 'aliases', 'accounting', 'records', 'financial'], 'aliases': ['accounting reports', 'audit records', 'financial statements'], 'user_queries': ['What is accounts and audit reports?', 'How does accounts and audit reports work?', 'Can you explain accounts and audit reports?']}, 'metadata': {'jurisdiction': 'India', 'act': 'Consumer Protection Act, 2019', 'language': 'en', 'review_status': 'draft', 'confidence': 0.0, 'created_by': 'KeywordAliasAgent', 'reviewed_by': None, 'version': 1}}
Concept ID:
alias.accounts_and_audi

### Converting chunks into documents


In [6]:
from langchain_core.documents import Document

In [9]:
class DocumentBuilder:

    def __init__(self, knowledge_cards):
        self.knowledge_cards = knowledge_cards

    def build_documents(self):
        documents = []

        for card in self.knowledge_cards:

            text = self._build_text(card)

            metadata = self._build_metadata(card)

            document = Document(
                page_content=text,
                metadata=metadata
            )

            documents.append(document)

        return documents
    def _build_text(self, card):

        parts = []

        # Title
        if card.get("title"):
            parts.append(
                f"Title: {card['title']}"
            )

        # Description
        if card.get("description"):
            parts.append(
                f"Description: {card['description']}"
            )

        # Content
        if card.get("content"):
            parts.append(
                "Content:\n" +
                self._format_value(card["content"])
            )

        # Search information
        search = card.get("search", {})

        if search.get("keywords"):
            parts.append(
                "Keywords: " +
                ", ".join(search["keywords"])
            )

        if search.get("aliases"):
            parts.append(
                "Aliases: " +
                ", ".join(search["aliases"])
            )

        if search.get("user_queries"):
            parts.append(
                "User Queries:\n" +
                "\n".join(
                    f"- {query}"
                    for query in search["user_queries"]
                )
            )

        return "\n\n".join(parts)
    def _format_value(self, value):

        if isinstance(value, dict):

            parts = []

            for key, val in value.items():

                formatted_key = key.replace("_", " ").title()

                parts.append(
                    f"{formatted_key}: "
                    f"{self._format_value(val)}"
                )

            return "\n".join(parts)

        elif isinstance(value, list):

            return "\n".join(
                f"- {self._format_value(item)}"
                for item in value
            )

        else:

            return str(value)
    def _build_metadata(self, card):

        metadata = card.get("metadata", {})

        return {
            "concept_id": card.get("concept_id"),
            "concept_type": card.get("concept_type"),
            "title": card.get("title"),

            "derived_from": str(
                card.get("derived_from", [])
            ),

            "related_concepts": str(
                card.get("related_concepts", [])
            ),

            "act": metadata.get("act"),
            "jurisdiction": metadata.get("jurisdiction"),
            "language": metadata.get("language"),
            "review_status": metadata.get("review_status"),
            "version": metadata.get("version")
        }

In [10]:
document_builder = DocumentBuilder(knowledge_cards)

documents = document_builder.build_documents()

print(f"Created {len(documents)} documents")

Created 4147 documents


### Verifying if I want this type of data or not


In [15]:
print(documents[0].page_content)
print(documents[0].metadata)

Title: Accounts and audit reports

Description: Alternative names and search aliases for: Accounts and audit reports

Content:
Aliases: - accounting reports
- audit records
- financial statements

Keywords: accounts, audit, reports, alternative, names, search, aliases, accounting, records, financial

Aliases: accounting reports, audit records, financial statements

User Queries:
- What is accounts and audit reports?
- How does accounts and audit reports work?
- Can you explain accounts and audit reports?
{'concept_id': 'alias.accounts_and_audit_reports', 'concept_type': 'alias', 'title': 'Accounts and audit reports', 'derived_from': '[]', 'related_concepts': "['evidence.accounts_audit_report']", 'act': 'Consumer Protection Act, 2019', 'jurisdiction': 'India', 'language': 'en', 'review_status': 'draft', 'version': 1}


### Creating EmbeddingsManager to manage embeddings


In [16]:
from langchain_huggingface import HuggingFaceEmbeddings

c:\Users\OMEN\Desktop\Codes\coding\python\rag-lawbot\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [17]:
class EmbeddingManager:

    def __init__(
        self,
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    ):
        self.model_name = model_name

        self.embedding_model = HuggingFaceEmbeddings(
            model_name=self.model_name,
            model_kwargs={
                "device": "cpu"
            },
            encode_kwargs={
                "normalize_embeddings": True
            }
        )

    def get_model(self):
        return self.embedding_model

In [18]:
embedding_manager = EmbeddingManager()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2428.60it/s]


In [19]:
test_text = documents[0].page_content

vector = embedding_manager.get_model().embed_query(test_text)

print(type(vector))
print(len(vector))
print(vector[:10])

<class 'list'>
384
[0.008340747095644474, -0.0016678179381415248, -0.06608124822378159, 0.04833563417196274, -0.04639498144388199, 0.020590445026755333, 0.08482342213392258, -0.0024883816950023174, -0.029437478631734848, -0.029928099364042282]


Storing it into Vector DataBase


In [20]:
from pathlib import Path
from langchain_chroma import Chroma

In [21]:
VECTOR_DB_PATH = Path("../data/chroma_db")

print(VECTOR_DB_PATH)

..\data\chroma_db


In [22]:
vector_store = Chroma.from_documents(
    documents=documents,
    embedding=embedding_manager.get_model(),
    collection_name="consumer_protection_act",
    persist_directory=str(VECTOR_DB_PATH)
)

In [23]:
results = vector_store.similarity_search(
    "I bought a defective laptop and I want my money back",
    k=5
)

for i, doc in enumerate(results, 1):
    print("=" * 50)
    print(i)
    print(doc.metadata.get("concept_id"))
    print(doc.metadata.get("concept_type"))
    print(doc.metadata.get("title"))

1
example.appeal_to_national_commission
example
Appeal to National Commission
2
example.refund_for_defective_goods
example
Refund for Defective Goods
3
right.right_to_refund
right
Right to Refund
4
remedy.refund
remedy
Refund of consideration
5
example.warranty_documents_sales_agreement
example
Proving Non-Conformity to Express Warranty


### Retriever


In [24]:
from typing import List, Tuple
from langchain_core.documents import Document


class RAGRetriever:

    def __init__(self, vector_store, k: int = 5):

        self.vector_store = vector_store
        self.k = k

    def retrieve(self, query: str) -> List[Document]:
        """
        Retrieve the top-k most relevant legal documents.
        """

        return self.vector_store.similarity_search(
            query,
            k=self.k
        )

    def retrieve_with_scores(
        self,
        query: str
    ) -> List[Tuple[Document, float]]:

        return self.vector_store.similarity_search_with_score(
            query,
            k=self.k
        )

    def get_context(self, query: str) -> str:
        """
        Convert retrieved documents into context
        that can later be passed to the LLM.
        """

        documents = self.retrieve(query)

        context_parts = []

        for i, doc in enumerate(documents, 1):

            context_parts.append(
                f"""
--- Knowledge Card {i} ---

Concept ID:
{doc.metadata.get("concept_id")}

Type:
{doc.metadata.get("concept_type")}

Title:
{doc.metadata.get("title")}

Content:
{doc.page_content}
"""
            )

        return "\n".join(context_parts)

In [28]:
retriever = RAGRetriever(
    vector_store=vector_store,
    k=5
)
retriever

In [27]:
query = "I bought a defective laptop and I want my money back"

results = retriever.retrieve(query)
results
context = retriever.get_context(query)

print(context)


--- Knowledge Card 1 ---

Concept ID:
example.appeal_to_national_commission

Type:
example

Title:
Appeal to National Commission

Content:
Title: Appeal to National Commission

Description: Illustrative example for: procedure.appeal_to_national_commission

Content:
Scenario: Rahul, a consumer from Mumbai, purchased a defective smartphone from a retailer. He filed a complaint with the State Commission, but the order was not in his favor. Rahul wants to appeal to the National Commission. He deposits fifty per cent of the amount ordered by the State Commission and files an appeal with the National Commission, attaching the bill, cash memo, and receipt for the smartphone.
Outcome: The National Commission will consider Rahul's appeal and may pass an order in his favor if it finds that the State Commission's order was incorrect. The National Commission may also direct the retailer to replace the defective smartphone or refund the amount paid by Rahul.

Keywords: National Commission, appeal,

In [33]:
query = "What is refund"

results = retriever.retrieve(query)

print("Number of results:", len(results))
results

Number of results: 5


[Document(id='74e66c20-475f-4840-a78b-c3f4ede78686', metadata={'jurisdiction': 'India', 'concept_type': 'timeline', 'concept_id': 'timeline.a_period', 'review_status': 'reviewed', 'language': 'en', 'version': 1, 'act': 'Consumer Protection Act, 2019', 'title': 'Refund time limit', 'related_concepts': "['exception.except_such_contest_lottery_game_of_chance_or_skill_as_may_b', 'evidence.test_results_records_of_testing', 'definition.unfair_trade_practice']", 'derived_from': "['CPA2019-CH1-S2-47']"}, page_content='Title: Refund time limit\n\nDescription: refund the consideration within the period stipulated in the bill or cash memo or receipt or in the absence of such stipulation, within a period\n\nContent:\nDuration: a period\nTrigger: Not specified in the Act\nExceptions: - Not specified in the Act\n\nKeywords: refund, time, limit, consideration, period, stipulated, bill, cash, memo, receipt\n\nAliases: promotion, sale, money back timeline, refund deadline, reimbursement period, return 

In [34]:
results = retriever.retrieve_with_scores(
    "What is refund"
)

for i, (doc, score) in enumerate(results, start=1):

    print(f"\n{'=' * 70}")
    print(f"RESULT #{i}")
    print(f"{'=' * 70}")

    print(f"Score       : {score:.4f}")
    print(f"Concept ID  : {doc.metadata.get('concept_id')}")
    print(f"Type        : {doc.metadata.get('concept_type')}")
    print(f"Title       : {doc.metadata.get('title')}")

    print("\nContent:")
    print(doc.page_content[:1000])


RESULT #1
Score       : 0.8424
Concept ID  : timeline.a_period
Type        : timeline
Title       : Refund time limit

Content:
Title: Refund time limit

Description: refund the consideration within the period stipulated in the bill or cash memo or receipt or in the absence of such stipulation, within a period

Content:
Duration: a period
Trigger: Not specified in the Act
Exceptions: - Not specified in the Act

Keywords: refund, time, limit, consideration, period, stipulated, bill, cash, memo, receipt

Aliases: promotion, sale, money back timeline, refund deadline, reimbursement period, return time frame

User Queries:
- What is refund time limit?
- How does refund time limit work?
- Can you explain refund time limit?

RESULT #2
Score       : 0.8740
Concept ID  : remedy.refund
Type        : remedy
Title       : Refund of consideration

Content:
Title: Refund of consideration

Description: Refund of the consideration paid for defective goods or deficient services

Content:
Remedy: Refu

### all the results print below


In [36]:
results = retriever.retrieve(
    "I bought a defected laptop online 6 months ago"
)

for i, doc in enumerate(results, start=1):
    print("\n" + "=" * 80)
    print(f"RESULT {i}")
    print("=" * 80)

    print("Concept ID:", doc.metadata.get("concept_id"))
    print("Type:", doc.metadata.get("concept_type"))
    print("Title:", doc.metadata.get("title"))

    print("\nFull Content:")
    print(doc.page_content)


RESULT 1
Concept ID: example.warranty_documents_sales_agreement
Type: example
Title: Proving Non-Conformity to Express Warranty

Full Content:
Title: Proving Non-Conformity to Express Warranty

Description: Illustrative example for: evidence.warranty_documents_sales_agreement

Content:
Scenario: Amit bought a laptop with a two-year warranty. After one year, the laptop's screen started flickering. Amit wants to prove that the laptop does not conform to the express warranty and claim a replacement.
Outcome: To prove that the laptop does not conform to the express warranty, Amit needs to provide the warranty documents and the sales agreement as evidence. The product manufacturer will be liable if the product does not conform to the express warranty or the terms and conditions of the contract.

Keywords: express warranty, product liability, consumer protection, warranty documents, sales agreement, product manufacturer

User Queries:
- What documents are required to prove non-conformity to